# Image MCQ Generator -- Schema-Driven, Batch-Based, Judge-Validated

Generates image-based MCQs from a schema.

**Each question has:**
- `instruction` -- e.g. 'Read the notice and choose the correct answer.'
- `image_content` -- text description of a real-world notice/sign (separate key)
- `question` -- the comprehension question about the notice
- `options` / `correct_answer` / `explanation`

**Flow:**
```
Schema + Examples
       |
EasyImageMCQAgent   -> generate batch -> DifficultyJudge -> RubricJudge -> QuestionStore
MediumImageMCQAgent -> generate batch -> DifficultyJudge -> RubricJudge -> QuestionStore
HardImageMCQAgent   -> generate batch -> DifficultyJudge -> RubricJudge -> QuestionStore
       |
ImageMCQGenerationResult (store + rejected + warnings)
```

Run all cells top-to-bottom. Edit Cell 10 to change the schema.

## Cell 1 -- Setup & Imports

In [1]:
import sys, os, json, warnings
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

warnings.filterwarnings('ignore')

# Locate project root (directory that contains utils.py)
_candidate = Path().resolve()
if (_candidate / 'utils.py').exists():
    PROJECT_ROOT = _candidate
elif (_candidate.parent / 'utils.py').exists():
    PROJECT_ROOT = _candidate.parent
else:
    raise RuntimeError(
        f'Cannot locate project root from {_candidate}. '
        'Open Jupyter from d:/Topin or d:/Topin/notebooks.'
    )

DATA_DIR      = PROJECT_ROOT / 'data' / 'image_mcq'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Auto-inject venv site-packages
_venv_sp = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'
if not _venv_sp.exists():
    _venv_sp = PROJECT_ROOT / '.venv' / 'lib' / 'site-packages'
if _venv_sp.exists() and str(_venv_sp) not in sys.path:
    sys.path.insert(0, str(_venv_sp))
    print(f'Injected venv site-packages: {_venv_sp}')

import dspy
from pydantic import BaseModel, Field

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Artifacts    : {ARTIFACTS_DIR}')
print(f'DSPy version : {dspy.__version__}')

Injected venv site-packages: D:\Topin\.venv\Lib\site-packages
Project root : D:\Topin
Data dir     : D:\Topin\data\image_mcq
Artifacts    : D:\Topin\artifacts
DSPy version : 3.1.3


## Cell 2 -- Configure DSPy (Mistral)

In [18]:
from utils import configure_dspy_from_env

lm = configure_dspy_from_env()
print(f'Active LM : {lm}')

Active LM : <dspy.clients.lm.LM object at 0x000002187FEAE590>


## Cell 3 -- Load Datasets

Loads `training_dataset_24_clean.json` and `eval_dataset_24_clean.json` from `data/image_mcq/`.
All rows are used as generation reference examples.

In [19]:
TRAIN_JSON = DATA_DIR / 'training_dataset_24_clean.json'
EVAL_JSON  = DATA_DIR / 'eval_dataset_24_clean.json'

def _load_json(json_path):
    with open(json_path, encoding='utf-8') as f:
        return json.load(f)

train_rows = _load_json(TRAIN_JSON)
eval_rows  = _load_json(EVAL_JSON)

# All rows are used as generation reference examples.
# The rubric judge's image_content_coherence criterion rejects incoherent outputs.
train_positive = train_rows
eval_positive  = eval_rows

print(f'Training : {len(train_rows)} rows loaded')
print(f'Eval     : {len(eval_rows)} rows loaded')
print(f'Sample IDs: {train_rows[0]["question_id"]} ... {train_rows[-1]["question_id"]}')

Training : 24 rows loaded
Eval     : 24 rows loaded
Sample IDs: Q1 ... Q24


## Cell 4 -- Input Models

- `SubtopicRequirement` -- per-subtopic counts for Easy / Medium / Hard
- `InputSchema` -- topic + list of subtopic requirements + generation constraints
- `ExampleImageMCQQuestion` -- one reference question with `instruction`, `image_content`, `question`
- `ExampleImageMCQQuestionSet` -- collection with `filter_examples()` to pick relevant references

In [20]:
class SubtopicRequirement(BaseModel):
    """Per-subtopic question counts for each of the 6 CEFR levels.

    CEFR mapping:
        A1, A2  ->  Easy
        B1, B2  ->  Medium
        C1, C2  ->  Hard
    """
    subtopic:  str
    a1_count:  int = 0
    a2_count:  int = 0
    b1_count:  int = 0
    b2_count:  int = 0
    c1_count:  int = 0
    c2_count:  int = 0

    @property
    def easy_count(self) -> int: return self.a1_count + self.a2_count
    @property
    def medium_count(self) -> int: return self.b1_count + self.b2_count
    @property
    def hard_count(self) -> int: return self.c1_count + self.c2_count
    @property
    def total(self) -> int: return self.easy_count + self.medium_count + self.hard_count


class GenerationConstraints(BaseModel):
    questions_per_iteration:       int = 5
    max_iterations_per_difficulty: int = 20


class InputSchema(BaseModel):
    topic:       str
    subtopics:   list[SubtopicRequirement]
    constraints: GenerationConstraints = Field(default_factory=GenerationConstraints)


class ExampleImageMCQQuestion(BaseModel):
    instruction:    str
    image_content:  str
    question:       str
    options:        list[str]
    correct_answer: str
    explanation:    str = ''
    difficulty:     str | None = None   # 'Easy', 'Medium', or 'Hard'
    cefr:           str | None = None   # 'A1', 'A2', 'B1', 'B2', 'C1', 'C2'
    subtopic:       str | None = None


class ExampleImageMCQQuestionSet(BaseModel):
    items: list[ExampleImageMCQQuestion] = Field(default_factory=list)

    def filter_examples(
        self,
        *,
        subtopic: str,
        difficulty: str,
        cefr: str | None = None,
    ) -> list[ExampleImageMCQQuestion]:
        """Priority 1: subtopic + CEFR. 2: subtopic + difficulty. 3: difficulty. 4: all."""
        if cefr:
            p1 = [q for q in self.items if q.cefr == cefr and q.subtopic in (None, subtopic)]
            if p1:
                return p1
        p2 = [q for q in self.items
              if q.difficulty in (None, difficulty) and q.subtopic in (None, subtopic)]
        if p2:
            return p2
        p3 = [q for q in self.items if q.difficulty in (None, difficulty)]
        if p3:
            return p3
        return self.items


# Build example set from all training rows (loaded in Cell 3)
example_questions = ExampleImageMCQQuestionSet(items=[
    ExampleImageMCQQuestion(
        instruction=r['instruction'],
        image_content=r['image_content'],
        question=r['question'],
        options=r['options'],
        correct_answer=r['correct_answer'],
    )
    for r in train_rows
])

print('Input models defined.')
print(f'ExampleImageMCQQuestionSet: {len(example_questions.items)} positive examples loaded')

Input models defined.
ExampleImageMCQQuestionSet: 24 positive examples loaded


## Cell 5 -- Output Models

- `ImageMCQItem` -- one accepted question (includes `instruction`, `image_content`, `question` as separate keys)
- `QuestionStore` -- accepted questions grouped by difficulty (easy/medium/hard)
- `ImageMCQGenerationResult` -- final output: store + rejected + warnings

In [21]:
class ImageMCQItem(BaseModel):
    question_number:   int
    topic:             str
    subtopic:          str | None
    target_cefr:       str
    target_difficulty: str
    instruction:       str
    image_content:     str   # text description of the notice/sign -- always a separate key
    question:          str
    options:           list[str]   # exactly 4
    correct_answer:    str
    explanation:       str


class QuestionStore(BaseModel):
    """Accepted questions grouped by difficulty band."""
    easy:   list[ImageMCQItem] = Field(default_factory=list)
    medium: list[ImageMCQItem] = Field(default_factory=list)
    hard:   list[ImageMCQItem] = Field(default_factory=list)

    def add(self, item: ImageMCQItem) -> None:
        d = item.target_difficulty.strip().lower()
        if d == 'easy':     self.easy.append(item)
        elif d == 'medium': self.medium.append(item)
        elif d == 'hard':   self.hard.append(item)
        else: raise ValueError(f'Unknown difficulty: {item.target_difficulty}')

    def get_used_image_contents(self) -> list[str]:
        """All accepted image_content values -- avoid generating duplicates."""
        return [q.image_content for q in self.easy + self.medium + self.hard]

    def count(self, difficulty: str) -> int:
        d = difficulty.strip().lower()
        if d == 'easy':   return len(self.easy)
        if d == 'medium': return len(self.medium)
        if d == 'hard':   return len(self.hard)
        raise ValueError(f'Unknown difficulty: {difficulty}')

    def count_by_cefr(self, cefr: str) -> int:
        return sum(1 for q in self.easy + self.medium + self.hard if q.target_cefr == cefr)

    def all_items(self) -> list[ImageMCQItem]:
        return self.easy + self.medium + self.hard


class ImageMCQGenerationResult(BaseModel):
    store:    QuestionStore
    rejected: list[dict] = Field(default_factory=list)
    warnings: list[str]  = Field(default_factory=list)


print('Output models defined.')
print('  QuestionStore: easy / medium / hard lists')
print('  QuestionStore.count_by_cefr("A1") counts accepted A1 questions')

Output models defined.
  QuestionStore: easy / medium / hard lists
  QuestionStore.count_by_cefr("A1") counts accepted A1 questions


## Cell 6 -- Image MCQ Generator Signature + Agent

**Typed batch approach:**
- Input: `ImageMCQGenerationRequest` (topic, CEFR, filtered examples, used image_contents, batch size)
- Output: `ImageMCQGenerationBatch` -> `list[ImageMCQGeneratedQuestion]`

One generator call produces a full batch. `image_content` is always a separate field.

In [22]:
class ImageMCQGenerationRequest(BaseModel):
    topic:                       str
    subtopic:                    str
    target_cefr:                 str
    target_difficulty:           str
    example_questions:           list[ExampleImageMCQQuestion]
    already_used_image_contents: list[str]   # avoid generating duplicate notices
    batch_size:                  int


class ImageMCQGeneratedQuestion(BaseModel):
    instruction:    str
    image_content:  str   # text description of a real-world notice/sign
    question:       str
    options:        list[str]   # exactly 4
    correct_answer: str
    explanation:    str


class ImageMCQGenerationBatch(BaseModel):
    questions: list[ImageMCQGeneratedQuestion]


class ImageMCQGeneratorSignature(dspy.Signature):
    """Generate a batch of image-based MCQ questions for the given topic and difficulty.

    Each question describes a real-world notice/sign as image_content (a text description),
    then asks a comprehension question with exactly 4 options.

    STRICT RULES:
    - image_content must logically support the question and correct_answer.
    - correct_answer must be copied VERBATIM from one of the options.
    - EXACTLY 4 options -- no more, no less.
    - Each image_content must differ from all entries in already_used_image_contents.
    - Align vocabulary and notice complexity to the target_cefr level.
    - image_content is always a separate field -- never merge it into question or instruction.
    """
    request: ImageMCQGenerationRequest = dspy.InputField(
        desc='Batch generation parameters: topic, CEFR, difficulty, examples, used image_contents'
    )
    output: ImageMCQGenerationBatch = dspy.OutputField(
        desc='Batch of generated image-based MCQ questions'
    )


class ImageMCQGeneratorAgent(dspy.Module):
    """Generates image MCQs and runs the quota loop inside forward().

    Dependencies injected at construction time:
      self.store            -- QuestionStore that accumulates accepted questions
      self.difficulty_judge -- DifficultyJudgeWrapper (batch)
      self.rubric_judge     -- RubricJudgeWrapper (batch)

    forward() takes per-call parameters and runs the loop until quota is met.
    """

    def __init__(
        self,
        *,
        store:            QuestionStore,
        difficulty_judge,
        rubric_judge,
    ):
        super().__init__()
        self.generate         = dspy.ChainOfThought(ImageMCQGeneratorSignature)
        self.store            = store
        self.difficulty_judge = difficulty_judge
        self.rubric_judge     = rubric_judge

    def forward(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleImageMCQQuestionSet,
        topic:             str,
        subtopic:          str | None,
        target_cefr:       str,
        target_difficulty: str,
        target_count:      int,
        rejected:          list,
        warnings:          list,
    ) -> None:
        """Run the generation loop for one CEFR level until quota is met.

        Each iteration:
          1. self.generate(request) -- LLM call -> ImageMCQGenerationBatch
          2. hard_validate_image_mcq() all questions in batch
          3. self.difficulty_judge(valid_items) -- batch LLM call
          4. self.rubric_judge(diff_passed)     -- batch LLM call
          5. self.store.add(accepted)
          -> Repeat until self.store.count_by_cefr(target_cefr) >= target_count
        """
        if target_count <= 0:
            return

        iteration = 0
        q_number  = len(self.store.all_items()) + 1

        print(f'\n[{target_difficulty} / {target_cefr}]  '
              f'target={target_count}  subtopic={subtopic!r}')

        while self.store.count_by_cefr(target_cefr) < target_count:
            iteration += 1
            if iteration > schema.constraints.max_iterations_per_difficulty:
                msg = (f'{target_difficulty}/{target_cefr}: max iterations '
                       f'({schema.constraints.max_iterations_per_difficulty}) reached. '
                       f'Accepted {self.store.count_by_cefr(target_cefr)}/{target_count}.')
                warnings.append(msg)
                print(f'  [WARNING] {msg}')
                break

            needed   = target_count - self.store.count_by_cefr(target_cefr)
            batch_sz = min(schema.constraints.questions_per_iteration, needed)
            relevant = example_questions.filter_examples(
                subtopic=subtopic or '',
                difficulty=target_difficulty,
                cefr=target_cefr,
            )
            request = ImageMCQGenerationRequest(
                topic=topic,
                subtopic=subtopic or '',
                target_cefr=target_cefr,
                target_difficulty=target_difficulty,
                example_questions=relevant,
                already_used_image_contents=self.store.get_used_image_contents(),
                batch_size=batch_sz,
            )

            # 1. Generate batch
            print(f'  [Iter {iteration}] Generating {batch_sz} questions...')
            try:
                pred  = self.generate(request=request)
                batch = pred.output.questions
            except Exception as e:
                rejected.append({
                    'stage': 'generation', 'cefr': target_cefr,
                    'difficulty': target_difficulty,
                    'iteration': iteration, 'error': str(e),
                })
                print(f'  [Gen Error] iter={iteration}: {e}')
                continue

            # 2. Hard-validate entire batch
            valid_items: list[ImageMCQItem] = []
            for q in batch:
                errors = hard_validate_image_mcq(q)
                if errors:
                    rejected.append({
                        'stage': 'hard_validate', 'cefr': target_cefr,
                        'difficulty': target_difficulty,
                        'iteration': iteration,
                        'image_content': q.image_content[:60], 'errors': errors,
                    })
                    print(f'  [Hard Fail] {errors}')
                    continue
                valid_items.append(ImageMCQItem(
                    question_number=q_number,
                    topic=topic,
                    subtopic=subtopic,
                    target_cefr=target_cefr,
                    target_difficulty=target_difficulty,
                    instruction=q.instruction,
                    image_content=q.image_content,
                    question=q.question,
                    options=q.options,
                    correct_answer=q.correct_answer,
                    explanation=q.explanation,
                ))
                q_number += 1

            if not valid_items:
                print(f'  [Skip] All {len(batch)} failed hard validation.')
                continue
            print(f'  [Hard]  {len(valid_items)}/{len(batch)} passed.')

            # 3. DifficultyJudge -- one batch call
            try:
                diff_results = self.difficulty_judge(
                    items=valid_items,
                    expected_difficulty=target_difficulty,
                )
            except Exception as e:
                rejected.append({'stage': 'difficulty_judge_error', 'error': str(e)})
                print(f'  [Diff Error] {e}')
                continue

            diff_passed: list[ImageMCQItem] = []
            for item, diff in zip(valid_items, diff_results):
                if diff.passed:
                    diff_passed.append(item)
                else:
                    rejected.append({
                        'stage': 'difficulty', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': diff.reason,
                        'image_content': item.image_content[:60],
                    })
                    print(f'  [Diff Fail] Q{item.question_number}: {diff.reason}')

            print(f'  [Diff]  {len(diff_passed)}/{len(valid_items)} passed.')
            if not diff_passed:
                continue

            # 4. RubricJudge -- one batch call
            try:
                rub_results = self.rubric_judge(items=diff_passed)
            except Exception as e:
                rejected.append({'stage': 'rubric_judge_error', 'error': str(e)})
                print(f'  [Rub Error] {e}')
                continue

            accepted_count = 0
            for item, rub in zip(diff_passed, rub_results):
                if not rub.passed:
                    rejected.append({
                        'stage': 'rubric', 'q': item.question_number,
                        'cefr': target_cefr, 'difficulty': target_difficulty,
                        'reason': rub.reason,
                        'image_content': item.image_content[:60],
                    })
                    print(f'  [Rub  Fail] Q{item.question_number}: {rub.reason}')
                    continue

                # 5. Save to store
                self.store.add(item)
                accepted_count += 1
                total = self.store.count_by_cefr(target_cefr)
                print(f'  [Accepted]  Q{item.question_number} '
                      f'({target_cefr}/{target_difficulty}) '
                      f'{total}/{target_count}')
                if total >= target_count:
                    break

            print(f'  [Rubric] {accepted_count}/{len(diff_passed)} passed.')


print('ImageMCQGeneratorAgent defined.')
print('  __init__(store, difficulty_judge, rubric_judge)')
print('  forward(schema, topic, target_cefr, target_count, ...)  -- quota loop')

ImageMCQGeneratorAgent defined.
  __init__(store, difficulty_judge, rubric_judge)
  forward(schema, topic, target_cefr, target_count, ...)  -- quota loop


## Cell 7 -- Load DifficultyJudge + RubricJudge

Typed models inline. Loads optimized agents from `artifacts/` if available.
**RubricJudge** adds `image_content_coherence` as the highest-priority criterion:
- `Incoherent` -> forces Fail
- `Partially Coherent` -> forces Revise

In [23]:
# DifficultyJudge models
class ImageMCQJudgeQuestion(BaseModel):
    question_id:       str
    instruction:       str
    image_content:     str
    question:          str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str


class DifficultyResult(BaseModel):
    question_id:          str
    predicted_cefr:       Literal['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    predicted_difficulty: Literal['Easy', 'Medium', 'Hard']


class DifficultyOutput(BaseModel):
    results: list[DifficultyResult]


class ImageMCQDifficultySignature(dspy.Signature):
    """Classify image-based MCQ questions by CEFR level and difficulty.
    Analyse vocabulary of image_content, question phrasing, and option difficulty.
    A1/A2 -> Easy | B1/B2 -> Medium | C1/C2 -> Hard.
    Return one DifficultyResult per question in the same order.
    """
    questions: list[ImageMCQJudgeQuestion] = dspy.InputField(
        desc='Image-based MCQ questions to classify'
    )
    output: DifficultyOutput = dspy.OutputField(
        desc='Classification results for all questions'
    )


class ImageMCQDifficultyAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(ImageMCQDifficultySignature)
    def forward(self, questions: list[ImageMCQJudgeQuestion]) -> dspy.Prediction:
        return self.judge(questions=questions)


# RubricJudge models
class ImageMCQRubricQuestion(BaseModel):
    question_id:       str
    instruction:       str
    image_content:     str
    question:          str
    options:           list[str]
    correct_answer:    str
    explanation:       str
    target_cefr:       str
    target_difficulty: str
    language_variant:  str


class ImageMCQRubricResult(BaseModel):
    question_id:                          str
    image_content_coherence:              Literal['Coherent', 'Partially Coherent', 'Incoherent']
    grammatical_accuracy:                 Literal['No Issues', 'Minor Issues', 'Major Issues']
    spelling:                             Literal['No Issues', 'Minor Issues', 'Major Issues']
    ambiguity:                            Literal['No Issue', 'Minor Issue', 'Major Issue']
    functionality_alignment:              Literal['Aligned', 'Partially Aligned', 'Not Aligned']
    instruction_clarity_appropriateness:  Literal['Clear', 'Needs Improvement', 'Unclear']
    academic_language_exam_acceptability: Literal['Acceptable', 'Needs Improvement', 'Not Acceptable']
    option_explanation_consistency:       Literal['Consistent', 'Inconsistent']
    readability:                          Literal['Good', 'Needs Improvement', 'Poor']
    formatting_spacing:                   Literal['No Issues', 'Minor Issues', 'Major Issues']
    punctuation:                          Literal['No Issues', 'Minor Issues', 'Major Issues']
    overall_decision:                     Literal['Pass', 'Revise', 'Fail']
    priority_reason:                      str
    revision_feedback:                    str


class ImageMCQRubricOutput(BaseModel):
    results: list[ImageMCQRubricResult]


class ImageMCQRubricSignature(dspy.Signature):
    """Evaluate image-based MCQ questions using the rubric.
    image_content_coherence is highest priority -- 'Incoherent' forces Fail.
    ambiguity is second priority -- 'Major Issue' forces Fail.
    Return one ImageMCQRubricResult per question in the same order.
    """
    questions: list[ImageMCQRubricQuestion] = dspy.InputField(
        desc='Image-based MCQ questions to evaluate'
    )
    output: ImageMCQRubricOutput = dspy.OutputField(
        desc='Rubric evaluation results for all questions'
    )


class ImageMCQRubricAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.judge = dspy.ChainOfThought(ImageMCQRubricSignature)
    def forward(self, questions: list[ImageMCQRubricQuestion]) -> dspy.Prediction:
        return self.judge(questions=questions)


# Load optimized agents from artifacts
DIFF_ARTIFACT = ARTIFACTS_DIR / 'image_mcq_difficulty_optimized.json'
RUB_ARTIFACT  = ARTIFACTS_DIR / 'image_mcq_rubric_optimized.json'

_diff_agent = ImageMCQDifficultyAgent()
if DIFF_ARTIFACT.exists():
    _diff_agent.load(str(DIFF_ARTIFACT))
    print(f'Loaded DifficultyJudge from: {DIFF_ARTIFACT.name}')
else:
    print(f'DifficultyJudge: using unoptimized agent (artifact not found)')

_rub_agent = ImageMCQRubricAgent()
if RUB_ARTIFACT.exists():
    _rub_agent.load(str(RUB_ARTIFACT))
    print(f'Loaded RubricJudge from   : {RUB_ARTIFACT.name}')
else:
    print(f'RubricJudge: using unoptimized agent (artifact not found)')


# Batch judge wrappers -- return SimpleNamespace(.passed, .reason)

class DifficultyJudgeWrapper:
    """Accepts a batch of ImageMCQItems, returns list[SimpleNamespace(.passed, .reason)]."""

    def __call__(self, *, items: list, expected_difficulty: str) -> list:
        questions = [
            ImageMCQJudgeQuestion(
                question_id=str(item.question_number),
                instruction=item.instruction,
                image_content=item.image_content,
                question=item.question,
                options=item.options,
                correct_answer=item.correct_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
            )
            for item in items
        ]
        pred = _diff_agent(questions=questions)
        results = []
        for res in pred.output.results:
            passed = res.predicted_difficulty.lower() == expected_difficulty.lower()
            results.append(SimpleNamespace(
                passed=passed,
                reason=f'predicted_cefr={res.predicted_cefr} predicted_difficulty={res.predicted_difficulty}',
            ))
        return results


class RubricJudgeWrapper:
    """Accepts a batch of ImageMCQItems, returns list[SimpleNamespace(.passed, .reason)]."""

    def __init__(self, language_variant: str = 'British English'):
        self.language_variant = language_variant

    def __call__(self, *, items: list) -> list:
        questions = [
            ImageMCQRubricQuestion(
                question_id=str(item.question_number),
                instruction=item.instruction,
                image_content=item.image_content,
                question=item.question,
                options=item.options,
                correct_answer=item.correct_answer,
                explanation=item.explanation,
                target_cefr=item.target_cefr,
                target_difficulty=item.target_difficulty,
                language_variant=self.language_variant,
            )
            for item in items
        ]
        pred = _rub_agent(questions=questions)
        return [
            SimpleNamespace(
                passed=res.overall_decision == 'Pass',
                reason=res.priority_reason,
            )
            for res in pred.output.results
        ]


difficulty_judge = DifficultyJudgeWrapper()
rubric_judge     = RubricJudgeWrapper(language_variant='British English')

print('\nBatch judge wrappers ready.')
print('  difficulty_judge(items=[...], expected_difficulty="Easy") -> list of .passed/.reason')
print('  rubric_judge(items=[...])                                 -> list of .passed/.reason')

DifficultyJudge: using unoptimized agent (artifact not found)
RubricJudge: using unoptimized agent (artifact not found)

Batch judge wrappers ready.
  difficulty_judge(items=[...], expected_difficulty="Easy") -> list of .passed/.reason
  rubric_judge(items=[...])                                 -> list of .passed/.reason


## Cell 8 -- Hard Validate Helper

Structural checks before sending to judges:
- `instruction`, `image_content`, `question` not empty
- Exactly 4 options
- `correct_answer` is one of the options
- `explanation` not empty

In [24]:
def hard_validate_image_mcq(q: ImageMCQGeneratedQuestion) -> list[str]:
    """Returns a list of error strings. Empty list = valid."""
    errors = []
    if not q.instruction or not q.instruction.strip():
        errors.append('instruction is empty')
    if not q.image_content or not q.image_content.strip():
        errors.append('image_content is empty')
    if not q.question or not q.question.strip():
        errors.append('question is empty')
    if not isinstance(q.options, list) or len(q.options) != 4:
        errors.append(
            f'Expected 4 options, got '
            f'{len(q.options) if isinstance(q.options, list) else type(q.options).__name__}'
        )
    if q.correct_answer not in q.options:
        errors.append(f'correct_answer "{q.correct_answer}" not found in options')
    if not q.explanation or not q.explanation.strip():
        errors.append('explanation is empty')
    return errors


print('hard_validate_image_mcq() defined.')
print('  Checks: instruction | image_content | question | 4 options | correct_answer in options | explanation')

hard_validate_image_mcq() defined.
  Checks: instruction | image_content | question | 4 options | correct_answer in options | explanation


## Cell 9 -- Per-Difficulty Agents

- `BaseDifficultyAgent` -- delegates to `ImageMCQGeneratorAgent.forward()`
- `EasyImageMCQAgent`   -- `difficulty_name='Easy'`
- `MediumImageMCQAgent` -- `difficulty_name='Medium'`
- `HardImageMCQAgent`   -- `difficulty_name='Hard'`

In [25]:
_CEFR_TO_DIFFICULTY = {
    'A1': 'Easy',  'A2': 'Easy',
    'B1': 'Medium', 'B2': 'Medium',
    'C1': 'Hard',  'C2': 'Hard',
}


class BaseDifficultyAgent:
    """Delegates to ImageMCQGeneratorAgent.forward(). Holds only generator ref."""
    difficulty_name: str = ''
    default_cefr:    str = ''

    def __init__(self, *, generator: ImageMCQGeneratorAgent):
        self.generator = generator

    def generate_questions(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleImageMCQQuestionSet,
        topic:             str,
        subtopic:          str | None,
        target_cefr:       str,
        target_count:      int,
        rejected:          list,
        warnings:          list,
    ) -> None:
        self.generator(
            schema=schema,
            example_questions=example_questions,
            topic=topic,
            subtopic=subtopic,
            target_cefr=target_cefr,
            target_difficulty=self.difficulty_name,
            target_count=target_count,
            rejected=rejected,
            warnings=warnings,
        )


class EasyImageMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Easy'
    default_cefr    = 'A2'

class MediumImageMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Medium'
    default_cefr    = 'B1'

class HardImageMCQAgent(BaseDifficultyAgent):
    difficulty_name = 'Hard'
    default_cefr    = 'C1'


print('Per-difficulty agents defined.')
print('  __init__(generator)  -- only holds ImageMCQGeneratorAgent reference')
print('  generate_questions() -- calls self.generator(...) -> ImageMCQGeneratorAgent.forward()')

Per-difficulty agents defined.
  __init__(generator)  -- only holds ImageMCQGeneratorAgent reference
  generate_questions() -- calls self.generator(...) -> ImageMCQGeneratorAgent.forward()


## Cell 10 -- Orchestrator

`ImageMCQGenerationOrchestrator.run()` iterates over each `SubtopicRequirement` and runs A1->A2->B1->B2->C1->C2 in sequence.

In [26]:
_CEFR_LEVELS = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']


class ImageMCQGenerationOrchestrator:
    """Creates and wires all components, then runs the generation loop.

    Ownership:
      self.store     -- QuestionStore (shared via ImageMCQGeneratorAgent)
      self.generator -- ImageMCQGeneratorAgent (owns store + judges)
      self.easy/medium/hard_agent -- per-difficulty wrappers
    """

    def __init__(
        self,
        *,
        difficulty_judge: DifficultyJudgeWrapper,
        rubric_judge:     RubricJudgeWrapper,
    ):
        self.store = QuestionStore()

        self.generator = ImageMCQGeneratorAgent(
            store=self.store,
            difficulty_judge=difficulty_judge,
            rubric_judge=rubric_judge,
        )

        self.easy_agent   = EasyImageMCQAgent(generator=self.generator)
        self.medium_agent = MediumImageMCQAgent(generator=self.generator)
        self.hard_agent   = HardImageMCQAgent(generator=self.generator)

        self._agents = {
            'Easy':   self.easy_agent,
            'Medium': self.medium_agent,
            'Hard':   self.hard_agent,
        }

    def run(
        self,
        *,
        schema:            InputSchema,
        example_questions: ExampleImageMCQQuestionSet,
    ) -> ImageMCQGenerationResult:
        rejected = []
        warnings = []

        for req in schema.subtopics:
            SEP = '=' * 60
            print('\n' + SEP)
            print(f'Topic: {schema.topic}  |  Subtopic: {req.subtopic}')
            print(f'  Easy:   A1={req.a1_count}  A2={req.a2_count}')
            print(f'  Medium: B1={req.b1_count}  B2={req.b2_count}')
            print(f'  Hard:   C1={req.c1_count}  C2={req.c2_count}')
            print(SEP)

            for cefr in _CEFR_LEVELS:
                count_attr   = cefr.lower() + '_count'
                target_count = getattr(req, count_attr)
                if target_count == 0:
                    continue

                difficulty = _CEFR_TO_DIFFICULTY[cefr]
                agent      = self._agents[difficulty]

                agent.generate_questions(
                    schema=schema,
                    example_questions=example_questions,
                    topic=schema.topic,
                    subtopic=req.subtopic,
                    target_cefr=cefr,
                    target_count=target_count,
                    rejected=rejected,
                    warnings=warnings,
                )

        return ImageMCQGenerationResult(
            store=self.store,
            rejected=rejected,
            warnings=warnings,
        )


print('ImageMCQGenerationOrchestrator defined.')
print('  __init__(difficulty_judge, rubric_judge)')
print('    -> creates QuestionStore')
print('    -> creates ImageMCQGeneratorAgent(store, difficulty_judge, rubric_judge)')
print('    -> creates Easy/Medium/HardImageMCQAgent(generator)')

ImageMCQGenerationOrchestrator defined.
  __init__(difficulty_judge, rubric_judge)
    -> creates QuestionStore
    -> creates ImageMCQGeneratorAgent(store, difficulty_judge, rubric_judge)
    -> creates Easy/Medium/HardImageMCQAgent(generator)


## Cell 11 -- Define Schema + Examples, Then Run

**Edit this cell** to change the topic, subtopics, and counts.

Training examples loaded automatically from `data/image_mcq/training_dataset_24_clean.csv` (Cell 4).

In [27]:
# Define schema -- edit topic/subtopic and counts here
schema = InputSchema(
    topic='Reading Notices and Signs',
    subtopics=[
        SubtopicRequirement(
            subtopic='Public Notices',
            a1_count=2,
            a2_count=2,
            b1_count=2,
            b2_count=2,
            c1_count=1,
            c2_count=0,
        )
    ],
    constraints=GenerationConstraints(
        questions_per_iteration=5,
        max_iterations_per_difficulty=20,
    ),
)

# Build orchestrator and run
# Orchestrator creates QuestionStore + ImageMCQGeneratorAgent internally.
# Only the judges (loaded in Cell 7) are passed in.
orchestrator = ImageMCQGenerationOrchestrator(
    difficulty_judge=difficulty_judge,
    rubric_judge=rubric_judge,
)
result = orchestrator.run(
    schema=schema,
    example_questions=example_questions,
)

print('\n' + '=' * 60)
print('GENERATION COMPLETE')
print('=' * 60)
print(f'Easy   accepted : {result.store.count("Easy")}')
print(f'Medium accepted : {result.store.count("Medium")}')
print(f'Hard   accepted : {result.store.count("Hard")}')
print(f'Total  accepted : {len(result.store.all_items())}')
print(f'Total  rejected : {len(result.rejected)}')
if result.warnings:
    print('\nWarnings:')
    for w in result.warnings:
        print(f'  {w}')


Topic: Reading Notices and Signs  |  Subtopic: Public Notices
  Easy:   A1=2  A2=2
  Medium: B1=2  B2=2
  Hard:   C1=1  C2=0

[Easy / A1]  target=2  subtopic='Public Notices'
  [Iter 1] Generating 2 questions...
  [Hard Fail] ['explanation is empty']
  [Hard Fail] ['explanation is empty']
  [Skip] All 2 failed hard validation.
  [Iter 2] Generating 2 questions...
  [Hard Fail] ['explanation is empty']
  [Hard Fail] ['explanation is empty']
  [Skip] All 2 failed hard validation.
  [Iter 3] Generating 2 questions...
  [Hard Fail] ['explanation is empty']
  [Hard Fail] ['explanation is empty']
  [Skip] All 2 failed hard validation.
  [Iter 4] Generating 2 questions...
  [Hard Fail] ['explanation is empty']
  [Hard Fail] ['explanation is empty']
  [Skip] All 2 failed hard validation.
  [Iter 5] Generating 2 questions...
  [Hard Fail] ['explanation is empty']
  [Hard Fail] ['explanation is empty']
  [Skip] All 2 failed hard validation.
  [Iter 6] Generating 2 questions...
  [Hard Fail] ['e

## Cell 12 -- Save Output

In [28]:
output = {
    'schema': schema.model_dump(),
    'summary': {
        'easy':   {'accepted': result.store.count('Easy'),
                   'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Easy')},
        'medium': {'accepted': result.store.count('Medium'),
                   'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Medium')},
        'hard':   {'accepted': result.store.count('Hard'),
                   'rejected': sum(1 for r in result.rejected if r.get('difficulty') == 'Hard')},
        'total_accepted': len(result.store.all_items()),
        'total_rejected': len(result.rejected),
    },
    'questions': {
        'easy':   [q.model_dump() for q in result.store.easy],
        'medium': [q.model_dump() for q in result.store.medium],
        'hard':   [q.model_dump() for q in result.store.hard],
    },
    'rejected': result.rejected,
    'warnings': result.warnings,
}

out_path = DATA_DIR / 'image_mcq_generator_output.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Output saved -> {out_path}')

# Verify image_content is a separate top-level key in each question
sample = result.store.all_items()
if sample:
    first = sample[0].model_dump()
    assert 'image_content' in first, 'ERROR: image_content missing as top-level key!'
    print('\nSample Q1:')
    for k, v in first.items():
        print(f'  {k}: {repr(v)[:80]}')

Output saved -> D:\Topin\data\image_mcq\image_mcq_generator_output.json

Sample Q1:
  question_number: 1
  topic: 'Reading Notices and Signs'
  subtopic: 'Public Notices'
  target_cefr: 'A2'
  target_difficulty: 'Easy'
  instruction: 'Read the notice and choose the correct answer.'
  image_content: 'A library notice says the building will be closed for maintenance tomorrow.'
  question: 'Why will the library be closed tomorrow?'
  options: ['For a holiday', 'For maintenance', 'For a special event', 'For a meeting']
  correct_answer: 'For maintenance'
  explanation: 'The notice clearly states that the library will be closed for maintenance.'
